# Lab: The Memory Pipeline

**Module 02b — Context Engineering**

## Objectives

By the end of this lab you will be able to:

1. **Extract** durable facts from a conversation with an LLM-driven filter.
2. **Consolidate** new facts against existing ones with CREATE / UPDATE / DELETE.
3. **Retrieve** memories with a weighted score over relevance, recency, and importance.
4. **A/B test** the agent with memory on vs. off, using the shared `eval_kit`.
5. **Refactor** the pipeline behind a `MemoryProvider` Protocol — the Hermes Agent pattern.
6. **Implement** async memory writes with a `flush()` barrier and measure the latency win.

## The Story

Sessions are the *log*. Memory is what you *learn* from the log. The slides framed this as LLM-driven ETL: extract → consolidate → retrieve. We're going to build all three stages against a small JSON store, prove the system is actually doing something with a memory-on vs memory-off A/B, and then refactor the whole thing behind the same `MemoryProvider` / `MemoryManager` abstraction that `NousResearch/hermes-agent` uses in production.

| Part | Topic |
|------|-------|
| 1 | Setup + JSON memory store |
| 2 | Extraction — filter the signal |
| 3 | Consolidation — CREATE / UPDATE / DELETE |
| 4 | Retrieval — `0.6 × relevance + 0.2 × recency + 0.2 × importance` |
| 5 | The A/B test — memory on vs memory off |
| 6 | `MemoryProvider` abstraction — swap backends without touching the agent loop |
| 7 | Async memory writes — fire-and-forget with a `flush()` barrier |


## Setup

We use `litellm` for chat + JSON-mode extraction, and `sentence-transformers/all-MiniLM-L6-v2` (downloaded once, runs locally) for the relevance similarity. The shared eval kit comes from `shared/eval_kit/` at the repo root — see its README for the full threading story.


In [ ]:
# !uv pip install litellm python-dotenv sentence-transformers numpy

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / 'shared'))

import json
import math
import os
import time
import uuid
from dataclasses import dataclass, field
from pathlib import Path

import litellm
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

from eval_kit.ab_test import run_ab
from eval_kit.judge import LLMJudge

load_dotenv()
CHAT_MODEL = os.getenv("CHAT_MODEL", "deepseek/deepseek-v4-flash:free")
EXTRACT_MODEL = os.getenv("EXTRACT_MODEL", "deepseek/deepseek-v4-flash:free")

assert os.getenv("OPENROUTER_API_KEY"), (
    "Set OPENROUTER_API_KEY in .env"
)

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"chat model: {CHAT_MODEL} | embedder: all-MiniLM-L6-v2")


### A minimal JSON memory store

Tiny on purpose. The point of the lab is the pipeline, not the database. Replace this with a vector DB or a real key-value store in production.


In [ ]:
MEM_PATH = Path.cwd() / "lab_memory.json"


@dataclass
class Memory:
    id: str
    fact: str
    importance: int           # 1..5, scored at extract time
    embedding: list[float]    # cached vector
    created_at: float
    updated_at: float


def _save(memories: list[Memory]) -> None:
    MEM_PATH.write_text(json.dumps([m.__dict__ for m in memories], ensure_ascii=False, indent=2))


def _load() -> list[Memory]:
    if not MEM_PATH.exists():
        return []
    return [Memory(**d) for d in json.loads(MEM_PATH.read_text())]


def reset_store() -> None:
    if MEM_PATH.exists():
        MEM_PATH.unlink()


def upsert(memory: Memory) -> None:
    store = _load()
    store = [m for m in store if m.id != memory.id]
    store.append(memory)
    _save(store)


def delete(memory_id: str) -> None:
    _save([m for m in _load() if m.id != memory_id])


def embed(text: str) -> list[float]:
    return embedder.encode(text, normalize_embeddings=True).tolist()


reset_store()
print(f"memory store reset at {MEM_PATH}")


---
## Part 2: Extraction

The slide's `EXTRACT_PROMPT` verbatim — filter for durable facts about the user that would help in a *future* session. Greetings, jokes, and transient state are deliberately dropped.

We extend the slide's prompt slightly: each fact now comes with an **importance** score (1–5). That score lives with the memory for retrieval later.


In [ ]:
from pydantic import BaseModel, Field

class ExtractedFact(BaseModel):
    fact: str = Field(description="A durable fact about the user.")
    importance: int = Field(description="Importance score from 1..5. 5 = stable goal/identity, 3 = lasting preference, 1 = one-off mention.")

class ExtractionResult(BaseModel):
    facts: list[ExtractedFact]

EXTRACT_PROMPT = """Read the conversation. Extract only durable facts about the user that
would be useful in a future session: preferences, goals, constraints, key context.

Ignore: greetings, jokes, transient state, restated facts."""


def extract_facts(events: list[dict]) -> list[dict]:
    """Run the LLM extractor over a conversation and return a list of fact dicts.

    TODO: implement this.
      1. Call litellm.completion with model=EXTRACT_MODEL, a system message of
         EXTRACT_PROMPT and a user message of json.dumps(events, ensure_ascii=False),
         temperature=0, and response_format=ExtractionResult (structured JSON mode).
      2. Parse: ExtractionResult.model_validate_json(response.choices[0].message.content).
      3. Return [fact.model_dump() for fact in parsed.facts].
    """
    raise NotImplementedError


> **Your turn.** Implement `extract_facts()` above — it's just the structured-output call described in the docstring. The **Solution** cell below is collapsed; try it first, then expand to compare. The demo cell after it prints what survived extraction.


In [ ]:
# @solution  — collapsed; expand to compare with your attempt above.
# Running it defines the working `extract_facts` so the rest of the lab still runs.
def extract_facts(events: list[dict]) -> list[dict]:
    response = litellm.completion(
        model=EXTRACT_MODEL,
        messages=[
            {"role": "system", "content": EXTRACT_PROMPT},
            {"role": "user", "content": json.dumps(events, ensure_ascii=False)},
        ],
        temperature=0,
        response_format=ExtractionResult,
    )
    parsed = ExtractionResult.model_validate_json(response.choices[0].message.content)
    return [fact.model_dump() for fact in parsed.facts]


In [ ]:
sample_events = [
    {"role": "user", "content": "Hey! I'm trying to plan a vegetarian dinner party for Friday."},
    {"role": "assistant", "content": "Fun! How many guests?"},
    {"role": "user", "content": "Six. Two are gluten-free, and my partner doesn't eat mushrooms."},
    {"role": "user", "content": "Also, I prefer Italian over Indian for hosting — easier on guests."},
]
extracted = extract_facts(sample_events)
for f in extracted:
    print(f)


> **Notice what dropped.** The greeting ("Hey!") and the assistant's clarifying question never become memories. Only user-attributable, future-relevant facts do.


---
## Part 3: Consolidation — CREATE / UPDATE / DELETE

The slide's worked example: three sequential turns about diet that look contradictory unless the consolidator reconciles them.

We use the LLM to decide which operation each new fact maps to. The prompt is a slimmer version of the slide's `CONSOLIDATE_PROMPT`.


In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

class MemoryOperation(BaseModel):
    op: Literal["CREATE", "UPDATE", "DELETE"] = Field(description="The operation to perform.")
    id: Optional[str] = Field(None, description="The ID of the existing memory to update or delete (required for UPDATE/DELETE).")
    fact: Optional[str] = Field(None, description="The content of the fact (required for CREATE/UPDATE).")
    importance: Optional[int] = Field(None, description="The importance score from 1..5 (required for CREATE/UPDATE).")

class ConsolidationResult(BaseModel):
    operations: list[MemoryOperation]

CONSOLIDATE_PROMPT = """You are reconciling new facts against an existing memory store.

Rules:
- CREATE if the topic is new.
- UPDATE if the new fact refines an existing one (same topic, more nuance).
- DELETE if the new fact directly contradicts an existing one (the old one is now wrong)."""


def consolidate(new_facts: list[dict]) -> list[dict]:
    existing = _load()
    existing_payload = [{"id": m.id, "fact": m.fact} for m in existing]

    response = litellm.completion(
        model=EXTRACT_MODEL,
        messages=[
            {"role": "system", "content": CONSOLIDATE_PROMPT},
            {
                "role": "user",
                "content": json.dumps(
                    {"existing": existing_payload, "new": new_facts}, ensure_ascii=False
                ),
            },
        ],
        temperature=0,
        response_format=ConsolidationResult,
    )
    parsed = ConsolidationResult.model_validate_json(response.choices[0].message.content)
    operations = [op.model_dump(exclude_none=True) for op in parsed.operations]
    apply_operations(operations)
    return operations


def apply_operations(operations: list[dict]) -> None:
    now = time.time()
    for op in operations:
        kind = op.get("op")
        if kind == "CREATE":
            upsert(
                Memory(
                    id=str(uuid.uuid4())[:8],
                    fact=op["fact"],
                    importance=int(op.get("importance", 3)),
                    embedding=embed(op["fact"]),
                    created_at=now,
                    updated_at=now,
                )
            )
        elif kind == "UPDATE":
            for m in _load():
                if m.id == op["id"]:
                    upsert(
                        Memory(
                            id=m.id,
                            fact=op["fact"],
                            importance=int(op.get("importance", m.importance)),
                            embedding=embed(op["fact"]),
                            created_at=m.created_at,
                            updated_at=now,
                        )
                    )
                    break
        elif kind == "DELETE":
            delete(op["id"])


def print_store(label: str) -> None:
    print(f"\n--- {label} ---")
    for m in _load():
        print(f"  [{m.id}] (imp={m.importance}) {m.fact}")


reset_store()

# Turn 1: vegetarian.
ops1 = consolidate([{"fact": "User is vegetarian.", "importance": 4}])
print_store("after turn 1 (CREATE expected)")

# Turn 2: actually eats fish (UPDATE — refines the dietary stance).
ops2 = consolidate([{"fact": "User eats fish but no other meat (pescatarian).", "importance": 4}])
print_store("after turn 2 (UPDATE expected)")

# Turn 3: back to strict vegetarian (DELETE the pescatarian fact, CREATE veg).
ops3 = consolidate([{"fact": "User is strictly vegetarian again — no fish.", "importance": 4}])
print_store("after turn 3 (DELETE + CREATE expected)")


> The LLM picks the operation. If the model picks wrong (e.g., CREATE where UPDATE was correct), you'll see two near-duplicate rows after Turn 2 — that's the "noisy contradictory log" the slide warns about. Re-running with a stronger judge model usually fixes it; for production you'd add a validator.


---
## Part 4: Retrieval — scored, not just relevant

The slide's headline formula:

```
score = 0.6 × relevance + 0.2 × recency + 0.2 × importance
```

- **relevance**: cosine similarity between the query embedding and the memory embedding.
- **recency**: exponential decay on `now - updated_at`, half-life 14 days.
- **importance**: the 1–5 score stored at extract time, normalised to 0..1.

**One catch before the weights mean anything.** Raw cosine relevance from a sentence-transformer sits around 0.1–0.7 and rarely reaches 1.0, while recency and `importance/5` routinely hit 1.0. Blend them as-is and a high-importance but topically *irrelevant* memory can outrank the relevant one — the exact opposite of what retrieval is for. So we **min-max normalise each component across the retrieved candidates first**, *then* apply the weights. (The seeded facts below make this live: one has `importance=5`.)

The weights themselves are starting points — the slide calls them out as defaults to tune. Don't ship them blindly.


In [ ]:
HALF_LIFE_SECONDS = 14 * 24 * 3600


def _cosine(a: list[float], b: list[float]) -> float:
    av, bv = np.array(a), np.array(b)
    return float(np.dot(av, bv) / (np.linalg.norm(av) * np.linalg.norm(bv) + 1e-9))


def _recency(updated_at: float) -> float:
    age = max(0.0, time.time() - updated_at)
    return 0.5 ** (age / HALF_LIFE_SECONDS)


def _minmax(values: list[float]) -> list[float]:
    """Rescale one component's raw scores to [0, 1] across the candidate set.

    This is the fix that makes the weights meaningful: raw cosine relevance and
    the [0,1] recency/importance components live on different scales, so we put
    each on a common scale *relative to the other candidates* before weighting.
    If every candidate ties (range ≈ 0), there is no signal to spread — treat
    them as equally strong on this component.
    """
    lo, hi = min(values), max(values)
    if hi - lo < 1e-9:
        return [1.0 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]


def retrieve(query: str, k: int = 3) -> list[tuple[Memory, float]]:
    """Return the top-k memories by 0.6·relevance + 0.2·recency + 0.2·importance.

    TODO: implement this.
      1. Embed the query; load all memories (return [] if the store is empty).
      2. Build three lists of RAW component scores, one per memory:
           relevance  -> _cosine(query_emb, m.embedding)
           recency    -> _recency(m.updated_at)
           importance -> m.importance / 5
      3. Min-max normalise EACH list across the candidates with _minmax(...).
      4. Combine per memory: score = 0.6*rel + 0.2*rec + 0.2*imp.
      5. Sort by score descending and return the top k as (Memory, score) pairs.
    """
    raise NotImplementedError


> **Your turn.** Fill in `retrieve()` above. The **Solution** cell below is collapsed — try it yourself first, then expand to compare. (Running the solution cell defines a working `retrieve`, so the demo and the rest of the lab still run even if you skip the exercise.)


In [ ]:
# @solution  — collapsed; expand to compare with your attempt above.
# Running it defines the working `retrieve` so the rest of the lab still runs.
def retrieve(query: str, k: int = 3) -> list[tuple[Memory, float]]:
    memories = _load()
    if not memories:
        return []
    q_emb = embed(query)

    # Raw components — one list per dimension, parallel to `memories`.
    relevance = [_cosine(q_emb, m.embedding) for m in memories]
    recency = [_recency(m.updated_at) for m in memories]
    importance = [m.importance / 5 for m in memories]

    # Put all three on a comparable [0, 1] scale ACROSS the candidates, then weight.
    relevance = _minmax(relevance)
    recency = _minmax(recency)
    importance = _minmax(importance)

    scored = [
        (m, 0.6 * rel + 0.2 * rec + 0.2 * imp)
        for m, rel, rec, imp in zip(memories, relevance, recency, importance)
    ]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


In [ ]:
# Seed the store with five facts of varying ages and importances.
reset_store()
apply_operations([
    {"op": "CREATE", "fact": "User is strictly vegetarian.", "importance": 5},
    {"op": "CREATE", "fact": "User prefers Italian cuisine when hosting.", "importance": 3},
    {"op": "CREATE", "fact": "User's partner doesn't eat mushrooms.", "importance": 4},
    {"op": "CREATE", "fact": "User asked once about a cocktail recipe.", "importance": 1},
    {"op": "CREATE", "fact": "User mentioned they have two gluten-free guests on Friday.", "importance": 3},
])

for query in [
    "What should I cook for my dinner party?",
    "Anything I should know about drinks?",
]:
    print(f"\nQ: {query}")
    for m, score in retrieve(query, k=3):
        print(f"  score={score:.3f}  [{m.id}] {m.fact}")


---
## Part 5: A/B test — does memory actually help?

The slide pointed at this experiment explicitly: run the same agent with and without injected memory, score the answers, and look at the delta. Anything else is leading-indicator only.

We use the shared `eval_kit` so the judge is the same primitive you'll see again in Modules 01 and 03.


In [ ]:
def agent_with_memory(query: str) -> str:
    retrieved = retrieve(query, k=3)
    memory_block = "\n".join(f"- {m.fact}" for m, _ in retrieved)
    system = (
        "You are a helpful cooking assistant. "
        "Use the user memory below to personalise advice when relevant.\n"
        f"Known about user:\n{memory_block}"
    )
    response = litellm.completion(
        model=CHAT_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": query}],
        temperature=0.2,
    )
    return response.choices[0].message.content


def agent_no_memory(query: str) -> str:
    response = litellm.completion(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful cooking assistant."},
            {"role": "user", "content": query},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


judge = LLMJudge(
    rubric=(
        "Does the candidate answer reflect the user's known preferences "
        "(strict vegetarian, no mushrooms for partner, two gluten-free guests, "
        "prefers Italian when hosting)? "
        "Score 1 if the answer is consistent with all *relevant* preferences for this query; "
        "0 if it suggests something that violates them."
    ),
    scale_max=1,
)


def score_fn(case: str, output: str) -> float:
    return judge.score(output, context=f"User query: {case}").score


cases = [
    "Plan a 3-course dinner for my Friday party.",
    "Suggest a starter that everyone at the table can eat.",
    "What's a good main course given my guests?",
    "Recommend a dessert.",
    "Should I serve wine pairings?",
]

result = run_ab(
    treatment_fn=agent_with_memory,
    control_fn=agent_no_memory,
    cases=cases,
    score_fn=score_fn,
)
print(result.summary())


> **Reading the output.** Positive `delta` means memory helped on average. A near-zero delta means either (a) the judge rubric is too loose to detect the difference, or (b) the control was already lucky. Both are real failure modes; in Module 03 we'll see calibration techniques that distinguish them.

---
## Part 6: `MemoryProvider` abstraction — the Hermes pattern

Open [`agent/memory_provider.py`](https://github.com/NousResearch/hermes-agent/blob/main/agent/memory_provider.py) and [`agent/memory_manager.py`](https://github.com/NousResearch/hermes-agent/blob/main/agent/memory_manager.py) in the Hermes Agent repo. Two abstractions stand out:

- **`MemoryProvider`** — an interface (Python `Protocol`) for the *whole* memory pipeline of one "kind" of memory. Hermes ships separate providers for `user` memory and `agent` memory, all behind the same shape.
- **`MemoryManager`** — owns one provider, exposes **lifecycle hooks** (`on_turn_start`, `on_session_switch`, `shutdown_all`) the agent loop calls at well-defined moments.

The payoff: the agent loop never reaches into a JSON file or a SQLite table directly. It only ever talks to `manager.on_turn_start(...)`. Swap the provider, swap the entire backend — the agent loop doesn't notice.

We'll refactor Parts 2–4 of *this* lab behind that interface, then prove the abstraction holds by swapping in a one-line `InMemoryProvider` for tests.


In [ ]:
from typing import Protocol


class MemoryProvider(Protocol):
    """Pluggable interface for one 'kind' of memory.

    A provider owns extraction, consolidation, persistence, and retrieval. The
    MemoryManager coordinates one (or more) of these without knowing how any of
    them stores facts — JSON, SQLite, vector DB, or remote service.
    """

    def extract(self, events: list[dict]) -> list[dict]: ...
    def consolidate(self, new_facts: list[dict]) -> list[dict]: ...
    def retrieve(self, query: str, k: int = 3) -> list[tuple[Memory, float]]: ...
    def sync(self) -> None: ...


class JSONMemoryProvider:
    """Concrete provider backed by the JSON store from Parts 1-4.

    Just wraps the existing module-level functions. No behavior change — this is
    the value of the Protocol: refactor the seams, not the algorithms.
    """

    def extract(self, events: list[dict]) -> list[dict]:
        return extract_facts(events)

    def consolidate(self, new_facts: list[dict]) -> list[dict]:
        return consolidate(new_facts)

    def retrieve(self, query: str, k: int = 3) -> list[tuple[Memory, float]]:
        return retrieve(query, k=k)

    def sync(self) -> None:
        pass  # JSON writes are eager — nothing to flush.


class InMemoryProvider:
    """Test double — same Protocol, facts live in a process-local list.

    The point isn't quality; it's the *swap*. A test can run the full agent
    loop against this provider without touching disk.
    """

    def __init__(self) -> None:
        self._facts: list[dict] = []

    def extract(self, events: list[dict]) -> list[dict]:
        return extract_facts(events)  # reuse the same LLM extractor

    def consolidate(self, new_facts: list[dict]) -> list[dict]:
        ops = []
        for f in new_facts:
            self._facts.append(dict(f))
            ops.append({"op": "CREATE", **f})
        return ops

    def retrieve(self, query: str, k: int = 3) -> list[tuple[Memory, float]]:
        # Trivial retrieval — most recent k facts, score=1.0. The MemoryManager
        # contract is unchanged; the strategy is the provider's business.
        now = time.time()
        out = []
        for i, f in enumerate(self._facts[-k:]):
            m = Memory(
                id=f"mem-{i}",
                fact=f["fact"],
                importance=int(f.get("importance", 3)),
                embedding=[],
                created_at=now,
                updated_at=now,
            )
            out.append((m, 1.0))
        return out

    def sync(self) -> None:
        pass


class MemoryManager:
    """Owns one MemoryProvider and exposes lifecycle hooks.

    The agent loop calls these at well-defined moments:
    - on_turn_start: fetch memories to inject into the next LLM call.
    - on_session_end: extract durable facts from the just-finished conversation
      and consolidate them into the store.
    """

    def __init__(self, provider: MemoryProvider) -> None:
        self.provider = provider

    def on_turn_start(self, user_query: str, k: int = 3) -> list[Memory]:
        return [m for m, _ in self.provider.retrieve(user_query, k=k)]

    def on_session_end(self, events: list[dict]) -> list[dict]:
        facts = self.provider.extract(events)
        if not facts:
            return []
        ops = self.provider.consolidate(facts)
        self.provider.sync()
        return ops


In [ ]:
# Re-seed the JSON store so we have something to retrieve against.
reset_store()
apply_operations([
    {"op": "CREATE", "fact": "User is strictly vegetarian.", "importance": 5},
    {"op": "CREATE", "fact": "User prefers Italian cuisine when hosting.", "importance": 3},
    {"op": "CREATE", "fact": "User's partner doesn't eat mushrooms.", "importance": 4},
])

# Build two managers wrapping different providers — same interface.
mgr_json = MemoryManager(JSONMemoryProvider())
mgr_test = MemoryManager(InMemoryProvider())

# Prime the test provider with one fact so retrieve() has something to return.
mgr_test.provider.consolidate([{"fact": "Test fact: user loves teaching examples.", "importance": 2}])

query = "What should I cook for the dinner party?"
for label, mgr in [("JSONMemoryProvider", mgr_json), ("InMemoryProvider", mgr_test)]:
    retrieved = mgr.on_turn_start(query, k=2)
    print(f"\n[{label}] retrieved {len(retrieved)} memories:")
    for m in retrieved:
        print(f"  [{m.id}] {m.fact[:70]}")

# The point: the agent loop above only knows MemoryManager. Swapping the
# provider beneath it is one line, and the agent code never changes.


### Read the real thing

Pair what you just wrote with these two files in [`NousResearch/hermes-agent`](https://github.com/NousResearch/hermes-agent):

- [`agent/memory_provider.py`](https://github.com/NousResearch/hermes-agent/blob/main/agent/memory_provider.py) — the Protocol shape (`get_user_memory`, `add_to_user_memory`, prefetch helpers).
- [`agent/memory_manager.py`](https://github.com/NousResearch/hermes-agent/blob/main/agent/memory_manager.py) — orchestration, the single-external-provider invariant, and the lifecycle hooks the agent loop calls.

Production additions you'll see that we deliberately skipped: **prefetch / queue-prefetch** (start retrieving for turn *N+1* while the LLM is still answering turn *N*), and **multi-provider coordination** (the manager owns one local + one external provider, merges results). The Protocol stays the same — the wiring under the manager is where the complexity lives.


---
## Part 7: Async memory writes — fire-and-forget with a `flush()` barrier

The slide's per-turn cycle diagram drew **Fetch** and **Prepare** as blocking, but **Upload** as async. We've been blocking on all three so far. Time to fix that.

The shape Hermes uses (and most production agents end up at):

- `on_session_end_async(events)` → schedules `extract` + `consolidate` on a background task. Returns immediately. The user never waits.
- `await flush()` → barrier the agent loop calls at the top of the next turn, ensuring the prior turn's writes have landed before this turn's `retrieve()` reads.

The pattern relies on the `MemoryProvider` interface we just built — only `MemoryManager` changes; the providers themselves don't know they're being driven asynchronously.

::: callout-note
**Why the `flush()` barrier?** Without it, two adjacent turns can race: turn N writes, turn N+1 reads — but the write task is still in flight, so the read sees the *old* store. The barrier costs nothing on average (the write usually finishes during the LLM call) but guarantees correctness when it doesn't.
:::


In [ ]:
import asyncio


class AsyncMemoryManager:
    """Same lifecycle hooks as MemoryManager, but writes go through asyncio.

    - on_session_end_async: fire-and-forget. The agent moves on immediately;
      the write runs in a background task.
    - flush(): explicit barrier the agent calls when it needs to read what
      it just wrote (typically at the top of the next turn).

    We use asyncio.to_thread because the underlying extract/consolidate are
    sync LLM calls. In production you'd swap in litellm.acompletion and
    drop the to_thread shim.

    The three async-specific methods are left for you to implement (TODOs). The
    sync helpers (_do_write / _sync_write) are given.
    """

    def __init__(self, provider: MemoryProvider) -> None:
        self.provider = provider
        self._pending: list[asyncio.Task] = []

    async def on_turn_start_async(self, user_query: str, k: int = 3) -> list[Memory]:
        # TODO: before reading, make sure any in-flight writes from prior turns
        # have landed — call the barrier (self.flush) — then return the memories
        # from self.provider.retrieve(user_query, k=k) (just the Memory objects).
        raise NotImplementedError

    def on_session_end_async(self, events: list[dict]) -> asyncio.Task:
        # TODO: schedule self._do_write(events) as a background task with
        # asyncio.create_task, append it to self._pending, and return it
        # immediately. Do NOT await it — that's the whole point.
        raise NotImplementedError

    async def _do_write(self, events: list[dict]) -> None:
        # Hop to a worker thread so the (sync) LLM call doesn't block the loop.
        await asyncio.to_thread(self._sync_write, events)

    def _sync_write(self, events: list[dict]) -> None:
        facts = self.provider.extract(events)
        if facts:
            self.provider.consolidate(facts)
            self.provider.sync()

    async def flush(self) -> None:
        # TODO: the barrier. await every not-yet-done task in self._pending
        # (asyncio.gather), then clear the list.
        raise NotImplementedError


> **Your turn.** Implement the three TODO methods above: `on_turn_start_async` (flush, then retrieve), `on_session_end_async` (fire-and-forget `create_task`), and `flush` (await pending). The **Solution** cell below is collapsed — try it first, then expand. Running the solution cell redefines `AsyncMemoryManager` with the working hooks so the timing harness below runs either way.

In [ ]:
# @solution  — collapsed; expand to compare with your attempt above.
# Running it redefines AsyncMemoryManager with the working hooks so Part 7 runs.
class AsyncMemoryManager(AsyncMemoryManager):  # subclass to keep the docstring + helpers
    async def on_turn_start_async(self, user_query: str, k: int = 3) -> list[Memory]:
        # Make sure any in-flight writes from prior turns have landed before we read.
        await self.flush()
        return [m for m, _ in self.provider.retrieve(user_query, k=k)]

    def on_session_end_async(self, events: list[dict]) -> asyncio.Task:
        task = asyncio.create_task(self._do_write(events))
        self._pending.append(task)
        return task

    async def flush(self) -> None:
        pending = [t for t in self._pending if not t.done()]
        if pending:
            await asyncio.gather(*pending)
        self._pending.clear()


In [ ]:
import asyncio
import time as _time

# Three synthetic sessions — each one ends with the user dropping a durable fact.
# We measure the wall-clock time the user perceives per session.
SESSIONS = [
    (
        [
            {"role": "user", "content": "I just adopted a cat — she needs a hypoallergenic diet."},
            {"role": "assistant", "content": "Got it — I'll keep that in mind."},
        ],
        "What's a hypoallergenic cat food brand you'd suggest?",
    ),
    (
        [
            {"role": "user", "content": "I usually run on Tuesdays and Thursdays after work."},
            {"role": "assistant", "content": "Noted."},
        ],
        "Help me plan my workouts for next week.",
    ),
    (
        [
            {"role": "user", "content": "We moved from Jeddah to Riyadh last month."},
            {"role": "assistant", "content": "Welcome to Riyadh."},
        ],
        "Any good Italian restaurants nearby?",
    ),
]

# A turn's *user-perceived* cost is fetch + generate-the-reply. The memory write
# (extract + consolidate, two LLM calls) is NOT something the user should wait on.
# We stand in for the reply with GEN_LATENCY, and for the gap where the user reads
# the reply and composes the next message with READ_LATENCY — the real window a
# background write gets to finish in. In production GEN_LATENCY is your litellm
# call and READ_LATENCY is genuine human think-time; neither is artificial.
GEN_LATENCY = 0.5   # generating + streaming one assistant reply (counted)
READ_LATENCY = 2.0  # user reads the reply + types the next one (NOT on the clock)


def measure_sync() -> float:
    """Each turn blocks on extract + consolidate before the user is freed."""
    reset_store()
    mgr = MemoryManager(JSONMemoryProvider())
    total = 0.0
    for events, query in SESSIONS:
        t0 = _time.perf_counter()
        _ = mgr.on_turn_start(query, k=2)   # fetch memories
        _time.sleep(GEN_LATENCY)            # generate + stream the reply
        mgr.on_session_end(events)          # ...then BLOCK the turn on the write
        total += _time.perf_counter() - t0  # the user waited for all three
        _time.sleep(READ_LATENCY)           # user reads the reply (not counted, no overlap)
    return total


async def measure_async() -> float:
    """The write is fired after the reply and finishes while the user reads."""
    reset_store()
    mgr = AsyncMemoryManager(JSONMemoryProvider())
    total = 0.0
    for events, query in SESSIONS:
        t0 = _time.perf_counter()
        _ = await mgr.on_turn_start_async(query, k=2)  # flush prior write (already done
                                                        # during last READ) + fetch
        await asyncio.sleep(GEN_LATENCY)               # generate + stream the reply
        total += _time.perf_counter() - t0             # user only waited for fetch + gen
        mgr.on_session_end_async(events)               # fire-and-forget the write...
        await asyncio.sleep(READ_LATENCY)              # ...it runs WHILE the user reads
    await mgr.flush()  # drain the final write; not on any user's clock
    return total


sync_total = measure_sync()
async_total = asyncio.run(measure_async())

print(f"sync  total user-perceived time: {sync_total:.2f}s")
print(f"async total user-perceived time: {async_total:.2f}s")
print(
    f"speedup: {sync_total / async_total:.2f}x — same extract+consolidate work, but it "
    f"overlaps the user reading the reply instead of blocking the turn. The next turn's "
    f"flush() then finds the write already done, so the barrier costs ~nothing."
)


### Read the real thing

The async-write pattern lives in [`agent/memory_manager.py`](https://github.com/NousResearch/hermes-agent/blob/main/agent/memory_manager.py). What we just built — fire-and-forget `create_task` + a `flush()` barrier at turn start — is the same shape Hermes uses, only Hermes adds: per-provider queues, retry/backoff, and a `shutdown_all()` that drains pending writes before the process exits.

What changed in *this* lab: nothing about the algorithms. Extraction, consolidation, retrieval, scoring — all identical to Parts 2–4. The only difference between Part 6 and Part 7 is **when** the writes happen relative to the agent's reply. Latency moves; correctness doesn't.

> The lesson: well-factored interfaces let you change *when* code runs without changing *what* it does. The Hermes Agent codebase is the worked example.


---
## Reflection

### Key Takeaways

| Concept | What you learned |
|---|---|
| **Extract / Consolidate / Retrieve** | Memory is LLM-driven ETL on top of the session log. |
| **CREATE / UPDATE / DELETE** | Without consolidation you accumulate a noisy contradictory log — *worse* than no memory. |
| **Weighted retrieval** | Relevance alone is insufficient; recency and importance are first-class. Tune the weights. |
| **A/B over headline metric** | Anything else is leading-indicator only. We used the shared `eval_kit` judge to make the comparison apples-to-apples. |
| **Provider abstraction** | The whole memory pipeline hides behind one Protocol; the agent loop only ever talks to `MemoryManager`. Swap providers, not agent code. |
| **Async writes + flush barrier** | The user never waits for `extract` + `consolidate`. The barrier costs nothing on average and guarantees next-turn reads see prior writes. |

### Connection to other modules

- The **judge** you just used is the same `LLMJudge` introduced in Module 01 Lab 4 for the Arabic-civics rubric, and the same primitive Module 03 Lab 2 will use to fact-check the newsroom Writer.
- The **A/B harness** is the simplest version of what Module 03 Session 02 will formalise as golden-dataset evaluation with CI gates.
- The **MemoryProvider / MemoryManager** pattern recurs across [`NousResearch/hermes-agent`](https://github.com/NousResearch/hermes-agent), LangGraph checkpoints, Vertex Agent Engine Memory Bank, and Mem0 — different syntax, same shape.

### Bonus

Add **provenance** (slide §F): a `source` field on each memory (`"bootstrapped"`, `"explicit"`, `"implicit"`, `"tool"`) and break ties in retrieval by highest-trust source.

Then add a **second provider** — an `AgentMemoryProvider` for things the agent learns about *itself* (which strategies worked, which tools were flaky) — and have `MemoryManager` own both. That's the structure Hermes uses to keep user memory and agent memory cleanly separated.
